In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
# load datasets
income_df = pd.read_csv('income_by_nta.csv')
bus_subway_df = pd.read_csv('bus_vs_subway_by_nta.csv')
vehicles_df = pd.read_csv('vehicles_stored_by_NTA.csv')
stops_df = pd.read_csv('stops_with_nta.csv')

lateness_files = [
    'weighted_lateness_bronx.csv',
    'weighted_lateness_brooklyn.csv',
    'weighted_lateness_manhattan.csv',
    'weighted_lateness_queens.csv',
    'weighted_lateness_si.csv'
]

lateness_parts = []
for file in lateness_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        df.columns = ['route_id', 'weighted_avg_lateness']
        lateness_parts.append(df)
    else:
        print(f"WARNING: {file} not found, skipping")

lateness_df = pd.concat(lateness_parts, ignore_index=True)

In [ ]:
# car score
vehicles_df = vehicles_df.rename(columns={'NTA': 'NTACode'})
vehicles_df = vehicles_df[['NTACode', '% Use Car']].rename(columns={'% Use Car': 'pct_use_car'})
vehicles_df['pct_use_car'] = pd.to_numeric(vehicles_df['pct_use_car'], errors='coerce')

low = vehicles_df['pct_use_car'].min()
high = vehicles_df['pct_use_car'].max()

vehicles_df['car_score'] = 1 - ((vehicles_df['pct_use_car'] - low) / (high - low)) #high car use = lower bus need

In [5]:
# reliability score
lateness_df['weighted_avg_lateness'] = lateness_df['weighted_avg_lateness'].abs()

gtfs_folders = ['gtfs_b', 'gtfs_bx', 'gtfs_m', 'gtfs_q', 'gtfs_si']

trips_list = []
stop_times_list = []

for folder in gtfs_folders:
    trips_path      = os.path.join(folder, 'trips.txt')
    stop_times_path = os.path.join(folder, 'stop_times.txt')
    if os.path.exists(trips_path):
        trips_list.append(
            pd.read_csv(trips_path, usecols=['route_id', 'trip_id'])
        )
    if os.path.exists(stop_times_path):
        stop_times_list.append(
            pd.read_csv(stop_times_path, usecols=['trip_id', 'stop_id'])
        )
 
all_trips      = pd.concat(trips_list,      ignore_index=True).drop_duplicates()
all_stop_times = pd.concat(stop_times_list, ignore_index=True).drop_duplicates()
 
# route_id -> trip_id -> stop_id
route_stops = all_trips.merge(all_stop_times, on='trip_id')
route_stops = route_stops[['route_id', 'stop_id']].drop_duplicates()
 
# Align types for join
stops_df['stop_id']    = stops_df['stop_id'].astype(str)
route_stops['stop_id'] = route_stops['stop_id'].astype(str)
route_stops['route_id'] = route_stops['route_id'].astype(str)
lateness_df['route_id'] = lateness_df['route_id'].astype(str)
 
# stop_id -> NTACode
route_nta = route_stops.merge(
    stops_df[['stop_id', 'NTACode']], on='stop_id', how='inner'
).drop_duplicates()
 
# Attach lateness to each route-NTA pair
lateness_nta = route_nta.merge(lateness_df, on='route_id', how='inner')
 
# Average lateness across all routes serving each NTA
nta_lateness = (
    lateness_nta
    .groupby('NTACode')['weighted_avg_lateness']
    .mean()
    .reset_index()
    .rename(columns={'weighted_avg_lateness': 'avg_lateness'})
)
 
low  = nta_lateness['avg_lateness'].min()
high = nta_lateness['avg_lateness'].max()
# Higher deviation from schedule = higher bus need
nta_lateness['reliability_score'] = (
    (nta_lateness['avg_lateness'] - low) / (high - low)
)

ValueError: No objects to concatenate

In [ ]:
merged = income_df[['NTACode', 'income_score']]

merged = merged.merge(vehicles_df[['NTACode', 'car_score']], on='NTACode', how='outer')
merged = merged.merge(bus_subway_df[['NTACode', 'bus_dependency_rate']].rename(columns={'bus_dependency_rate': 'bus_subway_score'}),on='NTACode', how='outer')
merged = merged.merge(nta_lateness[['NTACode', 'reliability_score']],on='NTACode', how='outer')
 
# placeholder... replace when Gina delivers ridership data
merged['ridership_score'] = np.nan

In [ ]:
# bus need index
# weights are arbitrary
W_INCOME      = 0.25
W_CAR         = 0.20
W_BUS_SUBWAY  = 0.20
W_RELIABILITY = 0.15
W_RIDERSHIP   = 0.20 #excluded until we get ridership data

available_weight = W_INCOME + W_CAR + W_BUS_SUBWAY + W_RELIABILITY

merged['index_score'] = (
    (W_INCOME      / available_weight) * merged['income_score'].fillna(0)    +
    (W_CAR         / available_weight) * merged['car_score'].fillna(0)       +
    (W_BUS_SUBWAY  / available_weight) * merged['bus_subway_score'].fillna(0)+
    (W_RELIABILITY / available_weight) * merged['reliability_score'].fillna(0)
).round(6)

In [ ]:
# sanity check
cols = ['income_score', 'car_score', 'bus_subway_score', 'reliability_score']
complete = merged[cols].notna().all(axis=1).sum()
 
print(f"Total NTAs: {len(merged)}")
print(f"NTAs with all 4 scores present: {complete}")
print(f"NTAs missing at least one score: {len(merged) - complete}")
 
print("\nTop 10 highest need NTAs:")
print(
    merged.sort_values('index_score', ascending=False)
    .head(10)[['NTACode'] + cols + ['index_score']]
    .to_string(index=False)
)
 
print("\nBottom 10 lowest need NTAs:")
print(
    merged.sort_values('index_score')
    .head(10)[['NTACode'] + cols + ['index_score']]
    .to_string(index=False)
)

In [ ]:
# export final dataset
merged[['NTACode', 'index_score']].rename(
    columns={'index_score': 'Score'}
).to_csv('bus_need_index.csv', index=False)
 
# full dataset for group reference and future merges
merged.to_csv('bus_need_index_full.csv', index=False)
 
print("\nExported:")